# Threat Code Processing

This notebook processes IUCN threat classification codes and creates a structured JSON mapping for the dashboard.

**Reference:** [IUCN Threat Classification Scheme](https://www.iucnredlist.org/resources/threat-classification-scheme) This reference is mentioned in the dataset.

**Output:** `data/processed/threat_codes.json`

In [1]:
import json
import pandas as pd

## Define IUCN Threat Codes

Based on the IUCN Red List threat classification scheme, mapping threat codes to human-readable descriptions.

In [2]:
threat_codes = {
    # 1. Residential & Commercial Development
    '1.1:DevResid': 'Housing & Urban Areas',
    '1.2:DevComInd': 'Commercial & Industrial Areas',
    '1.3:DevRnR': 'Tourism & Recreation Areas', 
    '1.4:DevUnspec': 'Development (Unspecified)',
    
    # 2. Agriculture & Aquaculture
    '2.1:AgNTC': 'Annual & Perennial Non-Timber Crops',
    '2.2:AgWood': 'Wood & Pulp Plantations',
    '2.3:AgLivestock': 'Livestock Farming & Ranching',
    '2.4:Aqua': 'Marine & Freshwater Aquaculture',
    '2.5:AgUnspec': 'Agriculture (Unspecified)',
    
    # 3. Energy Production & Mining
    '3.1:OilGas': 'Oil & Gas Drilling',
    '3.2:MinQuar': 'Mining & Quarrying',
    '3.3:Renew': 'Renewable Energy',
    '3.4:EnergUnspec': 'Energy Production (Unspecified)',
    
    # 4. Transportation & Service Corridors
    '4.1:RoadRail': 'Roads & Railroads',
    '4.2:UtilityLine': 'Utility & Service Lines',
    '4.3:ShipLane': 'Shipping Lanes',
    '4.4:FlightPath': 'Flight Paths',
    '4.5:UnspecLine': 'Transportation Corridors (Unspecified)',
    
    # 5. Biological Resource Use
    '5.1:UseAnimals': 'Hunting & Collecting Animals',
    '5.2:UsePlant': 'Gathering Terrestrial Plants',
    '5.3:UseWood': 'Logging & Wood Harvesting',
    '5.4:UseAquatic': 'Fishing & Harvesting Aquatic Resources',
    '5.5:UseUnspec': 'Biological Resource Use (Unspecified)',
    
    # 6. Human Intrusions & Disturbance
    '6.1:ActivRnR': 'Recreational Activities',
    '6.2:WarConflict': 'War, Civil Unrest & Military Exercises',
    '6.3:Other': 'Work & Other Activities',
    
    # 7. Natural System Modifications
    '7.1:FireSystem': 'Fire & Fire Suppression',
    '7.2:WaterSystem': 'Dams & Water Management/Use',
    '7.3:OtherSystem': 'Other Ecosystem Modifications',
    
    # 8. Invasive & Problematic Species
    '8.1:SDAlienInv': 'Invasive Non-Native/Alien Species',
    '8.2:SDNativProb': 'Problematic Native Species',
    '8.3:Genes': 'Introduced Genetic Material',
    '8.4:SDUnknownO': 'Problematic Species/Disease (Unknown Origin)',
    '8.5:ViralPrion': 'Viral/Prion-Induced Diseases',
    '8.6:DUnknownC': 'Disease (Unknown Cause)',
    
    # 9. Pollution
    '9.1:PollDWW': 'Domestic & Urban Waste Water',
    '9.2:PollIndMil': 'Industrial & Military Effluents',
    '9.3:PollAgrFor': 'Agricultural & Forestry Effluents',
    '9.4:PollGarbSW': 'Garbage & Solid Waste',
    '9.5:PollAir': 'Air-Borne Pollutants',
    '9.6:PollEnergy': 'Excess Energy (Light, Heat, Sound)',
    '9.7:PollUnspec': 'Pollution (Unspecified)',
    
    # 11. Climate Change & Severe Weather
    '11.1:CC_Habitat': 'Habitat Shifting & Alteration',
    '11.2:CC_Drought': 'Droughts',
    '11.3:CC_Temp': 'Temperature Extremes',
    '11.4:CC_WetWeather': 'Storms & Flooding',
    '11.5:CC_other': 'Climate Change (Other)',
    
    # 12. Other
    '12.0:Other': 'Other Threats (Unspecified)',
}

threat_categories = {
    '1': 'Residential & Commercial Development',
    '2': 'Agriculture & Aquaculture',
    '3': 'Energy Production & Mining',
    '4': 'Transportation & Service Corridors',
    '5': 'Biological Resource Use',
    '6': 'Human Intrusions & Disturbance',
    '7': 'Natural System Modifications',
    '8': 'Invasive & Problematic Species',
    '9': 'Pollution',
    '11': 'Climate Change & Severe Weather',
    '12': 'Other Threats',
}

print(f"Total threat codes defined: {len(threat_codes)}")
print(f"Total threat categories: {len(threat_categories)}")

Total threat codes defined: 48
Total threat categories: 11


## Validate Against Dataset

Check that all threat codes in our Grossi dataset have corresponding definitions.

In [3]:
# Load Grossi data
df_grossi = pd.read_csv('../data/processed/grossi_included_clean.csv')

# Extract all unique threat codes
threats = df_grossi['Threat'].dropna().astype(str)
all_codes = set()
for threat_string in threats:
    codes = threat_string.split(';')
    all_codes.update(codes)

print(f"Threat codes found in dataset: {len(all_codes)}\n")

# Validate coverage
missing = []
for code in sorted(all_codes):
    if code in threat_codes:
        print(f"✓ {code}: {threat_codes[code]}")
    else:
        print(f"✗ {code}: MISSING DEFINITION")
        missing.append(code)

if missing:
    print(f"\n  Missing {len(missing)} threat definitions")
else:
    print(f"\n All {len(all_codes)} codes have definitions")

Threat codes found in dataset: 27

✓ 1.1:DevResid: Housing & Urban Areas
✓ 1.2:DevComInd: Commercial & Industrial Areas
✓ 1.4:DevUnspec: Development (Unspecified)
✓ 11.1:CC_Habitat: Habitat Shifting & Alteration
✓ 11.3:CC_Temp: Temperature Extremes
✓ 2.1:AgNTC: Annual & Perennial Non-Timber Crops
✓ 2.2:AgWood: Wood & Pulp Plantations
✓ 2.3:AgLivestock: Livestock Farming & Ranching
✓ 2.5:AgUnspec: Agriculture (Unspecified)
✓ 3.1:OilGas: Oil & Gas Drilling
✓ 3.3:Renew: Renewable Energy
✓ 3.4:EnergUnspec: Energy Production (Unspecified)
✓ 4.1:RoadRail: Roads & Railroads
✓ 4.3:ShipLane: Shipping Lanes
✓ 4.5:UnspecLine: Transportation Corridors (Unspecified)
✓ 5.1:UseAnimals: Hunting & Collecting Animals
✓ 5.4:UseAquatic: Fishing & Harvesting Aquatic Resources
✓ 5.5:UseUnspec: Biological Resource Use (Unspecified)
✓ 6.3:Other: Work & Other Activities
✓ 8.1:SDAlienInv: Invasive Non-Native/Alien Species
✓ 9.1:PollDWW: Domestic & Urban Waste Water
✓ 9.2:PollIndMil: Industrial & Military Efflue

## Save to JSON

Export the threat code mappings to a structured JSON file for use in the dashboard.

In [4]:
# Prepare output structure
output = {
    "metadata": {
        "source": "IUCN Red List Threat Classification Scheme",
        "reference": "https://www.iucnredlist.org/resources/threat-classification-scheme",
        "total_codes": len(threat_codes),
        "total_categories": len(threat_categories)
    },
    "threat_codes": threat_codes,
    "threat_categories": threat_categories
}

# Save to JSON
output_path = '../data/processed/threat_codes.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f" Saved threat codes to: {output_path}")
print(f"   - {len(threat_codes)} threat codes")
print(f"   - {len(threat_categories)} threat categories")

 Saved threat codes to: ../data/processed/threat_codes.json
   - 48 threat codes
   - 11 threat categories
